# Prediction Check and Coverage on test dataset

In this notebook we perform prediction check on a test dataset for the trained posterior estimators to reproduce Figures 14 and 15 in Ronchi et al. (2026). In particular we consider these two experiments:

- When the entire observed X-ray population is considered (for Figure 14) we use the trained posterior estimator saved on the PIC server at the following path: `/data/magnesia/common/paper_ronchi_etal_2025/B_double_lognorm_dip-tor_heavy/tsnpe_experiment_1_maps8_res32/learning/models/SBI_ConvolutionMDN/20260115_142235/round_4`.
- When only the sample of young magnetars and XDINSs is considered for inference (for Figure 15) we use the trained posterior estimator saved on the PIC server at the following path: `/data/magnesia/common/paper_ronchi_etal_2025/B_double_lognorm_dip-tor_heavy/tsnpe_experiment_1_maps8_res32_youngxdins/learning/models/SBI_ConvolutionMDN/20260202_122351/round_4`.

Note that in order to make this notebook work, you first need to download the results data from `/data/magnesia/common/paper_ronchi_etal_2025/experiments_paper.zip` unpack the file and copy the entire folder experiments into `MAGNESIA_population_synthesis/data/paper_results/ronchi_etal_2026/experiments`. 

We also plot the coverage probability for the two trained posterior estimators to reproduce Figure 16.

In [ ]:
import collections
import corner
import json
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pathlib
import sys
import torch
import os
import shutil
from matplotlib.ticker import ScalarFormatter
from typing import Tuple

In [ ]:
# By default we consider the posterior inferred using entire X-ray sample.
# To consider the inference results using only young magnetars and XDINSs set `use_young_xdins_only` to True.
use_young_xdins_only = True

if use_young_xdins_only:
    stats_path = "../../data/paper_results/paper_ronchi_etal_2026/experiments/tsnpe_experiment_1_maps8_res32_youngxdins/data/statistics_train.json"
    directory_path = "../../data/paper_results/paper_ronchi_etal_2026/experiments/tsnpe_experiment_1_maps8_res32_youngxdins/learning/models/SBI_ConvolutionMDN/20260202_122351"
    test_data_path = f"../../data/paper_results/paper_ronchi_etal_2026/experiments/tsnpe_experiment_1_maps8_res32_youngxdins/inference/logs/SBI_ConvolutionMDN/20260218_143149/round_4/posterior_samples_test_data.npz"
    coverage_round = np.load(f"../../data/paper_results/paper_ronchi_etal_2026/experiments/tsnpe_experiment_1_maps8_res32_youngxdins/inference/logs/SBI_ConvolutionMDN/20260218_143149/round_4/coverage_probability.npy")
else:
    stats_path = "../../data/paper_results/paper_ronchi_etal_2026/experiments/tsnpe_experiment_1_maps8_res32/data/statistics_train.json"
    directory_path = "../../data/paper_results/paper_ronchi_etal_2026/experiments/tsnpe_experiment_1_maps8_res32/learning/models/SBI_ConvolutionMDN/20260115_142235"
    test_data_path = f"../../data/paper_results/paper_ronchi_etal_2026/experiments/tsnpe_experiment_1_maps8_res32/inference/logs/SBI_ConvolutionMDN/20260617_110628/round_4/posterior_samples_test_data.npz"
    coverage_round = np.load(f"../../data/paper_results/paper_ronchi_etal_2026/experiments/tsnpe_experiment_1_maps8_res32/inference/logs/SBI_ConvolutionMDN/20260617_110628/round_4/coverage_probability.npy")

In [ ]:
def import_statistics(stats_path: str) -> Tuple[list,list,list,list]:
    """
    Extracting the mean and standard deviation for all the parameters in the `stats_path` file.

    Args:
        stats_path (str): Path to the file where the statistics are saved.

    Returns:
        (Tuple[list,list,list,list]): Mean, standard deviation, maximun and minimun for the parameters in the `stats_path` file.
    """
    std_list = []
    mean_list = []
    max_list = []
    min_list = []

    with open(stats_path, "r") as json_file:
        data = json.load(json_file)

    for key, value in data.items():
        std_list.append(value["std"])
        mean_list.append(value["mean"])
        max_list.append(value["max"])
        min_list.append(value["min"])

    mean = np.array(mean_list)
    std = np.array(std_list)
    max_list = np.array(max_list)
    min_list = np.array(min_list)

    return mean, std, max_list, min_list

In [ ]:
mean, std, par_max, par_min = import_statistics(stats_path)

In [ ]:
data = np.load(test_data_path)
true_values = data["true_values"]
test_posterior_samples = data["posterior_samples"]

In [ ]:
test_posterior_samples = test_posterior_samples * std + mean
true_values = true_values * std + mean

In [ ]:
num_samples = test_posterior_samples.shape[0]
num_params = test_posterior_samples.shape[2]

print(num_samples, num_params)

# Defining the confidence interval.
ci_prob = 0.95

In [ ]:
parameter_ranges = [[-1.5, 0.5], [0.1, 1], [12, 13.5], [0.1, 1], [13.5, 14.0], [0.1, 1], [0.1, 1], [-2, 0], [24.0, 28.0], [0.1, 1]]
parameter_labels = [
    r"$\mu_{\log P}$", 
    r"$\sigma_{\log P}$", 
    r"$\mu_{\log B, 1}$", 
    r"$\sigma_{\log B, 1}$", 
    r"$\mu_{\log B, 2}$", 
    r"$\sigma_{\log B, 2}$", 
    r"$w_{\log B}$", 
    r"$a_{\rm late}$", 
    r"$\mu_{L0}$", 
    r"$\alpha_{L}$"
]

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(25, 10))
#axes = axes.flatten()  # makes indexing easier

for param_idx in range(num_params):

    row = param_idx // 5
    col = param_idx % 5
    ax = axes[row, col]

    # Extract true values
    true_vals = true_values[:, param_idx]
    medians = []
    lowers = []
    uppers = []

    for i in range(num_samples):
        posterior = test_posterior_samples[i, :, param_idx]

        median = np.median(posterior)

        lower_percentile = (1 - ci_prob) / 2 * 100
        upper_percentile = (1 + ci_prob) / 2 * 100

        lower = np.percentile(posterior, lower_percentile)
        upper = np.percentile(posterior, upper_percentile)

        medians.append(median)
        lowers.append(lower)
        uppers.append(upper)

    medians = np.array(medians)
    lowers = np.array(lowers)
    uppers = np.array(uppers)

    yerr = np.vstack([medians - lowers, uppers - medians])

    ax.scatter(
        true_vals,
        medians,
        color="green",
        marker="x",
        s=30,
        label="Posterior median",
        zorder=2
    )
    
    ax.errorbar(
        true_vals,
        medians,
        yerr=yerr,
        fmt="none",
        ecolor="lightblue",
        alpha=0.8,
        capsize=3,
        label=f"{ci_prob*100}% CI",
        zorder=1
    )

    ax.plot(
        [true_vals.min(), true_vals.max()],
        [true_vals.min(), true_vals.max()],
        "k--",
    )

    row = param_idx // 5
    col = param_idx % 5
    ax = axes[row, col]

    # --- your plotting code here ---

    # Set title
    ax.set_title(parameter_labels[param_idx], fontsize=20)

    # Only bottom row gets x-axis label
    if row == 1:
        ax.set_xlabel("True value", fontsize=20)
    else:
        ax.set_xlabel("")

    # Only left column gets y-axis label
    if col == 0:
        ax.set_ylabel("Predicted value", fontsize=20)
    else:
        ax.set_ylabel("")

    ax.tick_params(axis='both', which='major', labelsize=12)
    ax.grid(True)

# Optional: remove empty subplots if num_params < 10
for j in range(num_params, 10):
    fig.delaxes(axes[j])

# Single legend outside (cleaner)
#handles, labels = axes[0].get_legend_handles_labels()
#fig.legend(handles, labels, loc="center right", fontsize=14)

plt.tight_layout(rect=[0, 0, 0.92, 1])

if use_young_xdins_only:
    plt.savefig(f'plots/predictive_test_youngxdins.png',bbox_inches="tight")
else:
    plt.savefig(f'plots/predictive_test_full.png',bbox_inches="tight")
    
plt.show()

## Coverage probability

Plot the coverage probability for round 5 of both experiment to reproduce Figure 16 in Ronchi et al. (2026).

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))

ax.plot(
    credibility_level,
    credibility_level,
    linestyle="-",
    color="black",
    linewidth=2,
    alpha=1,
    rasterized=True,
    label=r"Well-calibrated",
)

ax.plot(
    credibility_level,
    coverage_round,
    linestyle="-",
    color='tab:blue',
    linewidth=3,
    rasterized=True
)

ax.grid(which='both')
ax.set_xlabel(r"Credibility level $1 - \alpha$", fontsize=SMALL_SIZE)
ax.set_ylabel(r"Coverage Probability", fontsize=SMALL_SIZE)
ax.set_xlim(0,1)
ax.set_ylim(0,1)

if use_young_xdins_only:
    plt.savefig(f'plots/coverage_plot_round4_youngxdins.png',bbox_inches="tight")
else:
    plt.savefig(f'plots/coverage_plot_round4_full.png',bbox_inches="tight")